# Module 4.2 — Chunking & Embeddings
**DeskMate SLM Curriculum · Phase 4 · Notebook 22**

Build a searchable vector index over DeskMate FAQ documents:
1. Chunk raw markdown docs using sentence-boundary + semantic strategies
2. Embed with `BAAI/bge-small-en-v1.5`
3. Index with FAISS (speed) and Chroma (metadata filtering)

Read `4.2_chunking_embeddings.md` for full theory.

---

## Step 0 — Install

In [ ]:
%%capture
!pip install -q sentence-transformers==3.1.1 faiss-cpu==1.8.0 chromadb==0.5.5 \
               rank-bm25==0.2.2 scikit-learn==1.5.2

In [ ]:
import re, json, pathlib, random
import numpy as np
import matplotlib.pyplot as plt
from sklearn.decomposition import PCA

SEED = 42
random.seed(SEED); np.random.seed(SEED)
print('Imports OK')

## Step 1 — Runtime & Paths

In [ ]:
try:
    import google.colab; RUNTIME = 'colab'
    PROJECT_ROOT = pathlib.Path('/content/slm')
except ImportError:
    try:
        import kaggle_secrets; RUNTIME = 'kaggle'
        PROJECT_ROOT = pathlib.Path('/kaggle/working/slm')
    except ImportError:
        RUNTIME = 'local'
        PROJECT_ROOT = pathlib.Path('.').resolve()

DATA_PROC  = PROJECT_ROOT / 'data' / 'processed'
FAISS_DIR  = PROJECT_ROOT / 'data' / 'faiss'
CHROMA_DIR = PROJECT_ROOT / 'data' / 'chroma_db'
FAQ_DIR    = PROJECT_ROOT / 'data' / 'faq'
for d in [DATA_PROC, FAISS_DIR, CHROMA_DIR, FAQ_DIR]:
    d.mkdir(parents=True, exist_ok=True)
print(f'Runtime: {RUNTIME}')

## Step 2 — Build DeskMate FAQ Corpus

15 synthetic FAQ articles covering all 5 DeskMate intents.
In production, replace these with your real documentation.

In [ ]:
FAQ_ARTICLES = [
    {
        'filename': 'account_login.md',
        'product': 'all',
        'section': 'account_access',
        'content': (
            '# Account Login & Access\n\n'
            '## Forgotten Password\n\n'
            'If you have forgotten your password, click the \'Forgot Password\' '
            'link on the login page. Enter your registered email address and we '
            'will send a password reset link within 2 minutes. Check your spam '
            'folder if it does not arrive.\n\n'
            '## Account Locked\n\n'
            'After 5 failed login attempts your account is automatically locked '
            'for 30 minutes as a security measure. To unlock immediately, use the '
            '\'Forgot Password\' flow or contact support via live chat.\n\n'
            '## Two-Factor Authentication\n\n'
            'DeskMate supports TOTP-based 2FA. Enable it under Settings > Security. '
            'If you lose access to your authenticator app, use one of the 10 backup '
            'codes provided at setup. If all backup codes are used, contact support '
            'with a government ID for manual verification.\n\n'
            '## SSO / SAML Login\n\n'
            'Enterprise customers can configure SAML 2.0 SSO under Settings > '
            'Security > SSO. Your IT administrator must provide the IdP metadata XML. '
            'SSO is available on the Business and Enterprise plans only.'
        )
    },
    {
        'filename': 'billing_refunds.md',
        'product': 'all',
        'section': 'billing_dispute',
        'content': (
            '# Billing, Payments & Refunds\n\n'
            '## Refund Policy\n\n'
            'DeskMate offers a 30-day money-back guarantee on all new subscriptions. '
            'To request a refund, go to Account > Billing > Request Refund. '
            'Refunds are processed within 5-7 business days to the original payment method.\n\n'
            '## Duplicate Charges\n\n'
            'If you have been charged twice for the same billing period, please '
            'contact support with your invoice numbers. We will investigate and '
            'refund the duplicate within 3 business days.\n\n'
            '## Changing Your Plan\n\n'
            'Upgrade or downgrade your plan at any time under Account > Subscription. '
            'Upgrades take effect immediately with prorated billing. Downgrades take '
            'effect at the end of the current billing period.\n\n'
            '## Invoice Downloads\n\n'
            'All invoices are available as PDFs under Account > Billing > Invoice History. '
            'Invoices are generated on the 1st of each month and emailed to the '
            'billing contact on file.'
        )
    },
    {
        'filename': 'technical_bugs.md',
        'product': 'DeskMate Pro',
        'section': 'technical_bug',
        'content': (
            '# Known Issues & Bug Reports\n\n'
            '## CSV Export Error (ERR-500)\n\n'
            'A known issue causes the CSV export to return a 500 error for datasets '
            'larger than 50,000 rows. Workaround: filter your dataset to under '
            '50,000 rows before exporting, or use the API export endpoint. '
            'Fix expected in version 4.3.0 (estimated 2 weeks).\n\n'
            '## Mobile App Crash on iOS 17\n\n'
            'Users on iOS 17.4 and earlier may experience a crash on launch. '
            'Update to DeskMate Mobile v4.2.1 or later from the App Store to resolve.\n\n'
            '## Reporting a Bug\n\n'
            'Use the in-app bug reporter (Help > Report a Bug) to attach logs '
            'automatically. Include steps to reproduce, expected behaviour, and '
            'actual behaviour. Our engineering team triages all reports within 24 hours.'
        )
    },
    {
        'filename': 'getting_started.md',
        'product': 'all',
        'section': 'usage_question',
        'content': (
            '# Getting Started with DeskMate\n\n'
            '## Inviting Team Members\n\n'
            'Go to Settings > Team > Invite Member. Enter the email address of the '
            'person you want to invite and select their role (Admin, Member, or Viewer). '
            'They will receive an invitation email within 5 minutes. '
            'Team members can be added up to the seat limit of your plan.\n\n'
            '## Workspace Setup\n\n'
            'Your workspace name and logo can be configured under Settings > Workspace. '
            'The workspace URL (yourname.deskmate.io) is set at creation and cannot be changed.\n\n'
            '## Integrations\n\n'
            'DeskMate integrates with Slack, Jira, Zendesk, and Salesforce. '
            'Connect integrations under Settings > Integrations. '
            'Each integration requires admin permissions in both systems.\n\n'
            '## API Access\n\n'
            'API keys are generated under Settings > Developer > API Keys. '
            'The REST API documentation is at docs.deskmate.io/api. '
            'Rate limits: 1,000 requests/minute on Pro, 10,000 on Enterprise.'
        )
    },
    {
        'filename': 'outage_status.md',
        'product': 'all',
        'section': 'outage_report',
        'content': (
            '# Service Status & Outages\n\n'
            '## Checking Current Status\n\n'
            'Real-time service status is available at status.deskmate.io. '
            'Subscribe to email or SMS alerts for automatic notifications '
            'when an incident is opened or resolved.\n\n'
            '## SLA Commitments\n\n'
            'DeskMate Pro: 99.9% monthly uptime SLA. '
            'DeskMate Enterprise: 99.99% monthly uptime SLA with dedicated support. '
            'SLA credits are issued automatically for qualifying downtime; '
            'no claim is required.\n\n'
            '## Reporting an Outage\n\n'
            'If status.deskmate.io does not reflect an issue you are experiencing, '
            'report it via support chat. Include your region, affected features, '
            'and the approximate start time.'
        )
    },
] * 3   # Repeat to reach 15 articles

# Assign unique filenames
for i, art in enumerate(FAQ_ARTICLES):
    suffix = f'_{i//5}' if i >= 5 else ''
    art['filename'] = art['filename'].replace('.md', f'{suffix}.md')
    (FAQ_DIR / art['filename']).write_text(art['content'])

print(f'Written {len(FAQ_ARTICLES)} FAQ articles to {FAQ_DIR}')

## Step 3 — Fixed-Size Chunker

Baseline: split on word count with overlap. Shows boundary artifacts.

In [ ]:
def fixed_chunk(text, chunk_size=256, overlap=32):
    words  = text.split()
    chunks, start = [], 0
    while start < len(words):
        end = min(start + chunk_size, len(words))
        chunks.append(' '.join(words[start:end]))
        if end == len(words): break
        start += chunk_size - overlap
    return chunks

sample_text = FAQ_ARTICLES[0]['content']
fixed_chunks = fixed_chunk(sample_text, chunk_size=80, overlap=10)
print(f'Fixed chunker: {len(fixed_chunks)} chunks')
print(f'\nChunk 0 (ends mid-sentence in some cases):')
print(fixed_chunks[0])
print(f'\nChunk 1 (overlap region):')
print(fixed_chunks[1][:200] if len(fixed_chunks) > 1 else '(only one chunk)')

## Step 4 — Semantic Chunker (Paragraph + Header Boundary)

DeskMate default: split on markdown headers and blank lines.

In [ ]:
def semantic_chunk(text, max_words=300):
    # Split on double newlines (paragraph boundary)
    paragraphs = re.split(r'\n{2,}', text.strip())
    chunks, buf, buf_len = [], [], 0
    for para in paragraphs:
        para_len = len(para.split())
        if buf_len + para_len > max_words and buf:
            chunks.append('\n\n'.join(buf))
            # Keep last paragraph as overlap
            buf     = [buf[-1]]
            buf_len = len(buf[0].split())
        buf.append(para)
        buf_len += para_len
    if buf:
        chunks.append('\n\n'.join(buf))
    return chunks

sem_chunks = semantic_chunk(sample_text, max_words=120)
print(f'Semantic chunker: {len(sem_chunks)} chunks')
for i, c in enumerate(sem_chunks):
    print(f'\n--- Chunk {i} ({len(c.split())} words) ---')
    print(c[:250])

## Step 5 — Chunk All FAQ Articles

In [ ]:
chunk_records = []

for art in FAQ_ARTICLES:
    chunks = semantic_chunk(art['content'], max_words=200)
    for j, chunk_text in enumerate(chunks):
        if len(chunk_text.split()) < 10:
            continue   # skip header-only fragments
        chunk_records.append({
            'id'      : f"{art['filename']}::chunk_{j}",
            'text'    : chunk_text,
            'source'  : art['filename'],
            'section' : art['section'],
            'product' : art['product'],
            'n_words' : len(chunk_text.split()),
        })

print(f'Total chunks: {len(chunk_records)}')
print(f'Mean words/chunk: {np.mean([r["n_words"] for r in chunk_records]):.0f}')
print(f'Min/Max: {min(r["n_words"] for r in chunk_records)} / '
       f'{max(r["n_words"] for r in chunk_records)}')

## Step 6 — Embed with `BAAI/bge-small-en-v1.5`

Note: chunk embeddings are NOT prefixed. Only queries use the prefix (Step 7).

In [ ]:
from sentence_transformers import SentenceTransformer

print('Loading embedding model ...')
embedder = SentenceTransformer('BAAI/bge-small-en-v1.5')

texts = [r['text'] for r in chunk_records]
print(f'Embedding {len(texts)} chunks ...')
embeddings = embedder.encode(
    texts,
    normalize_embeddings=True,
    batch_size=32,
    show_progress_bar=True,
)

print(f'\nEmbedding shape: {embeddings.shape}')
print(f'Sample norm: {np.linalg.norm(embeddings[0]):.4f}  (should be ~1.0 — L2-normalised)')

## Step 7 — Query Prefix Demo

`bge` models are asymmetrically trained: queries need a prefix, chunks do not.

In [ ]:
QUERY = 'How do I reset my password?'
BGE_PREFIX = 'Represent this sentence for searching relevant passages: '

q_no_prefix  = embedder.encode(QUERY, normalize_embeddings=True)
q_with_prefix = embedder.encode(BGE_PREFIX + QUERY, normalize_embeddings=True)

# Compare similarity to most relevant chunk
relevant_idx = next(i for i, r in enumerate(chunk_records)
                    if 'password' in r['text'].lower() or 'login' in r['text'].lower())
chunk_emb = embeddings[relevant_idx]

sim_no_prefix   = float(np.dot(q_no_prefix,   chunk_emb))
sim_with_prefix = float(np.dot(q_with_prefix, chunk_emb))
print(f'Query: "{QUERY}"')
print(f'Similarity WITHOUT prefix: {sim_no_prefix:.4f}')
print(f'Similarity WITH prefix   : {sim_with_prefix:.4f}')
print(f'Gain from prefix: {(sim_with_prefix - sim_no_prefix)*100:+.2f} percentage points')

## Step 8 — PCA Visualisation of Embedding Space

In [ ]:
pca    = PCA(n_components=2, random_state=SEED)
coords = pca.fit_transform(embeddings)

section_colours = {
    'account_access': '#4C72B0',
    'billing_dispute': '#DD8452',
    'technical_bug': '#55A868',
    'usage_question': '#C44E52',
    'outage_report': '#8172B2',
}
fig, ax = plt.subplots(figsize=(9, 6))
for section, colour in section_colours.items():
    mask = [r['section'] == section for r in chunk_records]
    xs   = coords[mask, 0]
    ys   = coords[mask, 1]
    ax.scatter(xs, ys, c=colour, label=section, alpha=0.75, s=60)
ax.set_title('FAQ Chunk Embeddings (PCA 2D) — clusters by intent')
ax.set_xlabel('PC1'); ax.set_ylabel('PC2')
ax.legend(loc='best', fontsize=8)
plt.tight_layout(); plt.show()
print(f'Explained variance: PC1={pca.explained_variance_ratio_[0]*100:.1f}%  '
       f'PC2={pca.explained_variance_ratio_[1]*100:.1f}%')

## Step 9 — Build FAISS Index

In [ ]:
import faiss

dim = embeddings.shape[1]
faiss_index = faiss.IndexFlatIP(dim)   # inner product = cosine (L2-normalised)
faiss_index.add(embeddings.astype('float32'))

print(f'FAISS IndexFlatIP: {faiss_index.ntotal} vectors  dim={dim}')

# Test search
SAMPLE_QUERIES = [
    'How do I reset my password?',
    'I was charged twice — can I get a refund?',
    'CSV export returns a 500 error',
    'How do I invite a team member?',
    'Is there a service outage right now?',
]

print('\nFAISS search results (top-3 per query):')
for q in SAMPLE_QUERIES:
    q_emb = embedder.encode(BGE_PREFIX + q, normalize_embeddings=True)
    D, I  = faiss_index.search(
        np.array([q_emb], dtype='float32'), k=3)
    print(f'\nQuery: {q}')
    for rank, (score, idx) in enumerate(zip(D[0], I[0])):
        r = chunk_records[idx]
        print(f'  [{rank+1}] score={score:.3f}  src={r["source"]}  '
               f'section={r["section"]}')
        print(f'      {r["text"][:100]}...')

## Step 10 — Build Chroma Index with Metadata

In [ ]:
import chromadb

chroma_client = chromadb.PersistentClient(path=str(CHROMA_DIR))
# Drop and recreate for idempotency
try:
    chroma_client.delete_collection('deskmate_faq')
except Exception:
    pass
collection = chroma_client.create_collection(
    'deskmate_faq',
    metadata={'hnsw:space': 'cosine'}
)

collection.add(
    documents=[r['text'] for r in chunk_records],
    embeddings=embeddings.tolist(),
    metadatas=[{'source': r['source'], 'section': r['section'],
                'product': r['product']} for r in chunk_records],
    ids=[r['id'] for r in chunk_records],
)
print(f'Chroma collection: {collection.count()} chunks')

## Step 11 — Metadata Filter Demo

Use the product field extracted by the Module 2.5 encoder to narrow retrieval.

In [ ]:
# Simulate: encoder extracted product='DeskMate Pro' from the ticket
query = 'The export button gives an error'
q_emb = embedder.encode(BGE_PREFIX + query, normalize_embeddings=True)

# Without filter
results_all = collection.query(
    query_embeddings=[q_emb.tolist()], n_results=3)
# With filter: product='DeskMate Pro' or 'all'
results_filtered = collection.query(
    query_embeddings=[q_emb.tolist()], n_results=3,
    where={'$or': [
        {'product': {'$eq': 'DeskMate Pro'}},
        {'product': {'$eq': 'all'}},
    ]}
)

print(f'Query: {query}')
print(f'\nWithout metadata filter:')
for doc, meta in zip(results_all['documents'][0], results_all['metadatas'][0]):
    print(f'  src={meta["source"]}  product={meta["product"]}')
    print(f'  {doc[:80]}...')
print(f'\nWith product filter (DeskMate Pro):')
for doc, meta in zip(results_filtered['documents'][0], results_filtered['metadatas'][0]):
    print(f'  src={meta["source"]}  product={meta["product"]}')
    print(f'  {doc[:80]}...')

## Step 12 — Save Index & Chunk Records

In [ ]:
import os

# FAISS index
faiss_path = FAISS_DIR / 'deskmate_faq.index'
faiss.write_index(faiss_index, str(faiss_path))

# Chunk records (text + metadata; used by BM25 in Module 4.3)
records_path = DATA_PROC / 'chunk_records.jsonl'
with open(records_path, 'w') as f:
    for r in chunk_records:
        f.write(json.dumps(r) + '\n')

# Index stats
faiss_size = os.path.getsize(faiss_path)
chroma_size = sum(
    os.path.getsize(os.path.join(dp, fn))
    for dp, _, files in os.walk(CHROMA_DIR)
    for fn in files
)
print(f'chunk_records.jsonl : {records_path}  ({os.path.getsize(records_path)/1e3:.0f} KB)')
print(f'FAISS index         : {faiss_path}  ({faiss_size/1e3:.0f} KB)')
print(f'Chroma DB           : {CHROMA_DIR}  ({chroma_size/1e3:.0f} KB)')
print(f'\nTotal chunks indexed: {len(chunk_records)}')
print(f'Embedding dimension : {dim}')

## Checkpoint

> **How do chunk size and overlap trade off recall against noise?**

Write your answer below.

In [ ]:
answer = """
[Write your answer here]
"""
print(answer)

## Deliverable Summary

| Artifact | Location |
|---|---|
| Chunk records (text + metadata) | `data/processed/chunk_records.jsonl` |
| FAISS index | `data/faiss/deskmate_faq.index` |
| Chroma DB | `data/chroma_db/` |

**What you've built:** a searchable vector index over the DeskMate FAQ.
FAISS for fast retrieval; Chroma for metadata-filtered retrieval using the product/version fields extracted by the Module 2.5 encoder.

**Next:** Module 4.3 — Retrieval that actually works.
Dense-only retrieval misses keyword-specific queries. Hybrid BM25 + dense search
with a cross-encoder reranker closes most of the gap — measured by hit-rate@3.